# Proyecto final: enfermedad cardiaca (Heart Disease)

Punto de partida común para **todas las parejas** en el proyecto final.
Las prácticas guiadas siguen usando *churn*.

Consulta `../proyecto-final.md` y `../data/alt/README.md`: datos UCI Cleveland,
codificaciones originales y `target = 1` para presencia de enfermedad.
Para el proyecto tendrás que adaptar `data.py`, `schemas.py` y la configuración.
Este notebook no sustituye el pipeline ejecutable, el batch, la API ni el informe
de drift. Los ausentes se imputan dentro del pipeline, solo con entrenamiento.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMILLA = 42
df = pd.read_csv("../data/alt/heart_disease.csv")
print(df.shape)
df.head()

In [ ]:
NUMERICAS = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
X, y = df[NUMERICAS + CATEGORICAS], df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA
)

In [ ]:
modelo = Pipeline(
    [
        (
            "preprocesamiento",
            ColumnTransformer(
                [
                    (
                        "num",
                        Pipeline(
                            [
                                ("imputar", SimpleImputer(strategy="median")),
                                ("escalar", StandardScaler()),
                            ]
                        ),
                        NUMERICAS,
                    ),
                    (
                        "cat",
                        Pipeline(
                            [
                                ("imputar", SimpleImputer(strategy="most_frequent")),
                                ("codificar", OneHotEncoder(handle_unknown="ignore")),
                            ]
                        ),
                        CATEGORICAS,
                    ),
                ]
            ),
        ),
        (
            "clasificador",
            RandomForestClassifier(n_estimators=200, random_state=SEMILLA),
        ),
    ]
)
modelo.fit(X_train, y_train)
proba_test = modelo.predict_proba(X_test)[:, 1]
print(f"ROC AUC test: {roc_auc_score(y_test, proba_test):.3f}")